# Exportación ONNX - Modelo B Aumentado Config 2

Este notebook exporta el **preprocesador y clasificador con datos aumentados** (config 2).

**Especificaciones:**
- Modelo: `best_model_b_augmented_config2.pth` (datos de entrenamiento aumentados)
- Config: 2 (aumentación aplicada)
- Parámetros sincronizados: n_fft=1024, n_mels=128, hop_length=256

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
from pathlib import Path

# Función auxiliar
def find_project_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "src").is_dir() and (p / "settings.gradle.kts").exists():
            return p
    raise FileNotFoundError("No encontrado")

project_root = find_project_root(Path.cwd()).resolve()
modelos_dir = project_root / "modelos"

COMMANDS = ["yes", "no", "up", "down", "left", "right", "on", "off", "stop", "go"]
NUM_CLASSES = len(COMMANDS)
IMG_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ✅ PARÁMETROS CORRECTOS - Sincronizados con proy1.ipynb
SAMPLE_RATE = 16000
N_FFT = 1024
HOP_LENGTH = 256
N_MELS = 128
F_MIN = 0.0
F_MAX = 8000.0
EPS = 1e-9

print(f"✅ Parámetros correctos:")
print(f"   • n_fft={N_FFT}, n_mels={N_MELS}, hop_length={HOP_LENGTH}")
print(f"   • Modelo: best_model_b_augmented_config2.pth (DATOS AUMENTADOS)")
print(f"   • Device: {DEVICE}")

✅ Parámetros correctos:
   • n_fft=1024, n_mels=128, hop_length=256
   • Modelo: best_model_b_augmented_config2.pth (DATOS AUMENTADOS)
   • Device: cuda


In [2]:
class AudioPreprocessorONNX(nn.Module):
    def __init__(self):
        super().__init__()
        self.register_buffer("window", torch.hann_window(N_FFT, dtype=torch.float32))
        fb = torch.from_numpy(
            torchaudio.functional.melscale_fbanks(
                n_freqs=(N_FFT // 2) + 1,
                f_min=F_MIN,
                f_max=F_MAX,
                n_mels=N_MELS,
                sample_rate=SAMPLE_RATE,
                norm="slaney",
                mel_scale="htk"
            ).numpy()
        ).transpose(0, 1)
        self.register_buffer("fb", fb.float())
    
    def forward(self, waveform):
        stft = torch.stft(
            waveform,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            win_length=N_FFT,
            window=self.window,
            center=True,
            pad_mode="reflect",
            return_complex=True,
        )
        power = (stft.real**2 + stft.imag**2).clamp_min(0.0)
        mel = torch.matmul(self.fb, power)
        mel = torch.log(mel + EPS)
        
        batch_size = mel.shape[0]
        for b in range(batch_size):
            mean = mel[b].mean()
            std = mel[b].std().clamp_min(EPS)
            mel[b] = (mel[b] - mean) / std
        
        mel = mel.unsqueeze(1)
        mel = F.interpolate(mel, size=(IMG_SIZE, IMG_SIZE), mode="bilinear", align_corners=False)
        return mel

print("✅ Preprocesador definido")

✅ Preprocesador definido


In [3]:
preprocessor = AudioPreprocessorONNX().to(DEVICE)
preprocessor.eval()
dummy_audio = torch.randn(1, SAMPLE_RATE, dtype=torch.float32, device=DEVICE)
output_path = modelos_dir / "audio_preprocessor_augmented.onnx"

print(f"\n🔧 Exportando preprocesador de audio...")
torch.onnx.export(
    preprocessor,
    (dummy_audio,),
    str(output_path),
    input_names=["audio"],
    output_names=["mel_spectrogram"],
    dynamic_axes={
        "audio": {0: "batch_size"},
        "mel_spectrogram": {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

print(f"✅ Preprocesador exportado: {output_path}")
print(f"   Tamaño: {output_path.stat().st_size / 1024:.1f} KB")


🔧 Exportando preprocesador de audio...


C:\Users\boyfa\AppData\Local\Temp\ipykernel_2956\1819729402.py:7: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0505 13:11:08.593000 2956 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `AudioPreprocessorONNX()` with `torch.export.export(..., strict=False)`...


W0505 13:11:10.267000 2956 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


[torch.onnx] Obtain model graph for `AudioPreprocessorONNX()` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Users\boyfa\miniconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).
Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, 

[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Preprocesador exportado: C:\AI_Proyecto1_2026\modelos\audio_preprocessor_augmented.onnx
   Tamaño: 22.6 KB


In [4]:
class ConvBNReLU(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, stride=1, padding=0, groups=1):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding, groups=groups, bias=False)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu6 = nn.ReLU6(inplace=True)

    def forward(self, x):
        x = self.conv(x)
        x = self.bn(x)
        x = self.relu6(x)
        return x

class InvertedResidual(nn.Module):
    def __init__(self, in_channels, out_channels, stride, expand_ratio):
        super().__init__()
        self.stride = stride
        self.use_res_connect = (stride == 1 and in_channels == out_channels)
        hidden_dim = int(round(in_channels * expand_ratio))
        layers = []
        if expand_ratio != 1:
            layers.append(ConvBNReLU(in_channels, hidden_dim, kernel_size=1))
        layers.append(ConvBNReLU(hidden_dim, hidden_dim, kernel_size=3, stride=stride, padding=1, groups=hidden_dim))
        layers.append(nn.Conv2d(hidden_dim, out_channels, kernel_size=1, stride=1, padding=0, bias=False))
        layers.append(nn.BatchNorm2d(out_channels))
        self.conv = nn.Sequential(*layers)

    def forward(self, x):
        if self.use_res_connect:
            return x + self.conv(x)
        else:
            return self.conv(x)

class MobileNetV2Audio(nn.Module):
    def __init__(self, num_classes=10, width_mult=1.0):
        super().__init__()
        block_settings = [(1,16,1,1), (6,24,2,2), (6,32,3,2), (6,64,4,2), (6,96,3,1), (6,160,3,2), (6,320,1,1)]
        in_channels = 32
        last_channels = 1280
        self.first_layer = ConvBNReLU(1, in_channels, kernel_size=3, stride=2, padding=1)
        features = []
        for t, c, n, s in block_settings:
            out_channels = int(c * width_mult)
            for i in range(n):
                stride = s if i == 0 else 1
                features.append(InvertedResidual(in_channels, out_channels, stride, t))
                in_channels = out_channels
        self.features = nn.Sequential(*features)
        self.last_layer = ConvBNReLU(in_channels, last_channels, kernel_size=1)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.dropout = nn.Dropout(p=0.2)
        self.classifier = nn.Linear(last_channels, num_classes)
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.first_layer(x)
        x = self.features(x)
        x = self.last_layer(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.dropout(x)
        x = self.classifier(x)
        return x

print("✅ Arquitecturas definidas")

✅ Arquitecturas definidas


In [5]:
# ── CARGAR MODELO B AUMENTADO CONFIG 2 ──────────────────────────

model_path = modelos_dir / "best_model_b_augmented_config2.pth"
onnx_path = modelos_dir / "modelo_b_augmented_config2.onnx"

print(f"\n📂 Buscando modelo aumentado:")
print(f"   Ruta: {model_path}")
print(f"   Existe: {model_path.exists()}")

if not model_path.exists():
    raise FileNotFoundError(f"❌ Modelo no encontrado: {model_path}")

classifier = MobileNetV2Audio(num_classes=NUM_CLASSES)
classifier.load_state_dict(torch.load(str(model_path), map_location="cpu"))
classifier = classifier.to(DEVICE)
classifier.eval()

print(f"\n✅ Modelo cargado: best_model_b_augmented_config2.pth")

dummy_mel = torch.randn(1, 1, IMG_SIZE, IMG_SIZE, dtype=torch.float32, device=DEVICE)

print(f"\n🔧 Exportando clasificador (datos aumentados)...")
torch.onnx.export(
    classifier,
    (dummy_mel,),
    str(onnx_path),
    input_names=["input"],
    output_names=["logits"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
    opset_version=17,
    do_constant_folding=True,
)

print(f"✅ Clasificador exportado: {onnx_path}")
print(f"   Tamaño: {onnx_path.stat().st_size / 1024:.1f} KB")


📂 Buscando modelo aumentado:
   Ruta: C:\AI_Proyecto1_2026\modelos\best_model_b_augmented_config2.pth
   Existe: True

✅ Modelo cargado: best_model_b_augmented_config2.pth

🔧 Exportando clasificador (datos aumentados)...


C:\Users\boyfa\AppData\Local\Temp\ipykernel_2956\1833878104.py:23: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(
W0505 13:11:11.555000 2956 site-packages\torch\onnx\_internal\exporter\_compat.py:133] Setting ONNX exporter to use operator set version 18 because the requested opset_version 17 is a lower version than we have implementations for. Automatic version conversion will be performed, which may not be successful at converting to the requested version. If version conversion is unsuccessful, the opset version of the exported model will be kept at 18. Please consider setting opset_version >=18 to leverage latest ONNX features


[torch.onnx] Obtain model graph for `MobileNetV2Audio([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `MobileNetV2Audio([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


c:\Users\boyfa\miniconda3\Lib\copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)
The model version conversion is not supported by the onnxscript version converter and fallback is enabled. The model will be converted using the onnx C API (target version: 17).


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅


Failed to convert the model to the target version 17 using the ONNX C API. The model was not modified
Traceback (most recent call last):
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 137, in call
    converted_proto = _c_api_utils.call_onnx_api(
        func=_partial_convert_version, model=model
    )
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\_c_api_utils.py", line 65, in call_onnx_api
    result = func(proto)
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnxscript\version_converter\__init__.py", line 132, in _partial_convert_version
    return onnx.version_converter.convert_version(
           ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
        proto, target_version=self.target_version
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "c:\Users\boyfa\miniconda3\Lib\site-packages\onnx\version_converter.py", line 39, in convert_version
    converted_model_str = C.convert_ve

[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
✅ Clasificador exportado: C:\AI_Proyecto1_2026\modelos\modelo_b_augmented_config2.onnx
   Tamaño: 235.9 KB


In [6]:
print("\n" + "="*70)
print("✅ EXPORTACIÓN COMPLETADA - MODELO B AUMENTADO CONFIG 2")
print("="*70)

files_to_check = [
    (modelos_dir / "audio_preprocessor_augmented.onnx", "Preprocesador Aumentado"),
    (modelos_dir / "modelo_b_augmented_config2.onnx", "Clasificador B Aumentado Config 2"),
]

print("\n📦 Archivos generados:")
for file_path, description in files_to_check:
    if file_path.exists():
        size_kb = file_path.stat().st_size / 1024
        print(f"✅ {description:35} | {size_kb:8.1f} KB | {file_path.name}")
    else:
        print(f"❌ {description:35} | NO ENCONTRADO")

print("\n📝 Parámetros de audio sincronizados:")
print(f"  • Sample Rate:    {SAMPLE_RATE} Hz")
print(f"  • N_FFT:          {N_FFT}")
print(f"  • Hop Length:     {HOP_LENGTH}")
print(f"  • N Mels:         {N_MELS}")
print(f"  • Output Size:    {IMG_SIZE}x{IMG_SIZE}")
print(f"  • Classes:        {NUM_CLASSES}")

print("\n🔄 Modelo de entrenamiento:")
print(f"  • Archivo: best_model_b_augmented_config2.pth")
print(f"  • Tipo: MobileNetV2 con datos AUMENTADOS")
print(f"  • Config: 2")

print("\n" + "="*70)
print("✅ Listo para copiar a app/src/main/assets/ en Android Studio")
print("="*70)


✅ EXPORTACIÓN COMPLETADA - MODELO B AUMENTADO CONFIG 2

📦 Archivos generados:
✅ Preprocesador Aumentado             |     22.6 KB | audio_preprocessor_augmented.onnx
✅ Clasificador B Aumentado Config 2   |    235.9 KB | modelo_b_augmented_config2.onnx

📝 Parámetros de audio sincronizados:
  • Sample Rate:    16000 Hz
  • N_FFT:          1024
  • Hop Length:     256
  • N Mels:         128
  • Output Size:    128x128
  • Classes:        10

🔄 Modelo de entrenamiento:
  • Archivo: best_model_b_augmented_config2.pth
  • Tipo: MobileNetV2 con datos AUMENTADOS
  • Config: 2

✅ Listo para copiar a app/src/main/assets/ en Android Studio
